# Fase 1.2 — Variable objetivo, población e indeterminados

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

El objetivo de la tesis es predecir si un crédito de libranza llega a **90 días de mora o más en los 12 meses siguientes al desembolso**. FINTECH no guarda los días de mora históricos, así que los reconstruyo día a día a partir del plan de pagos y de los pagos recibidos (módulo `src/tesis_v2/mora.py`). Es una reimplementación auditada del método de la v1, con los mismos parámetros, para poder compararla crédito a crédito.

En este notebook:

1. reconstruyo la mora y clasifico cómo terminó cada crédito;
2. armo el embudo de población y los conteos por objetivo (90+/12m, 60+/12m, 30+/6m);
3. resuelvo la discrepancia `>` frente a `≥` entre el texto y el código;
4. analizo los indeterminados y marco los créditos 131 y 250 para verificación;
5. comparo con el objetivo de la v1 y con una fuente independiente (`bd_pagosempresas`);
6. mido la sensibilidad a la tolerancia de pago y a la regla de refinanciación no enlazada (D4);
7. construyo el objetivo de supervivencia y una muestra estratificada para que yo la valide a mano con la operación.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

from tesis_v2 import config, estadistica as est, figuras, io_datos, objetivo
from tesis_v2.io_datos import llave, numero

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 30)
figuras.estilo()
config.asegurar_carpetas()

R = objetivo.cargar_y_reconstruir(guardar_series=True)
t = R["objetivo"]
grupo = io_datos.leer_parquet(config.PRIVADO / "grupo_cliente_v2.parquet")
t = t.merge(grupo, on="cli", how="left")
print(f"Fecha de corte (último pago observado): {R['corte'].date()}")
print(f"Créditos no anulados: {len(R['base'])} | reconstruidos: {len(t)} | pagos duplicados exactos eliminados: {R['n_pagos_dup']}")
print(f"Refinanciaciones con enlace explícito en bd_solicitud: {len(R['refi'])}")

Fecha de corte (último pago observado): 2026-10-08
Créditos no anulados: 1161 | reconstruidos: 1161 | pagos duplicados exactos eliminados: 0
Refinanciaciones con enlace explícito en bd_solicitud: 158


## 1. Cómo terminó cada crédito

El tipo de cierre importa porque define hasta cuándo observo la mora. Un crédito refinanciado se observa hasta el día de la refinanciación. Uno que terminó de pagar, hasta su último pago. Uno activo, hasta la fecha de corte. Los cierres "con saldo pendiente" son los delicados: el crédito figura como finalizado pero quedó debiendo.

In [2]:
cierres = pd.crosstab(t["cierre_v2"], t["anio_cohorte"], margins=True, margins_name="Total")
io_datos.guardar_resultado(cierres.reset_index(), "fase1", "cierres_por_cohorte")
cierres

anio_cohorte,2021,2022,2023,2024,2025,2026,Total
cierre_v2,,,,,,,
ACTIVO,0,2,5,28,56,301,392
A_TERMINO,2,16,52,56,55,3,184
CIERRE_ANTICIPADO_SIN_EXPLICACION,0,0,0,1,1,3,5
CON_SALDO_CLIENTE_RECURRENTE,1,8,40,33,1,0,83
INDETERMINADO,0,5,11,26,1,0,43
PREPAGO,1,13,47,74,55,15,205
REFINANCIACION,0,0,6,46,86,13,151
REFINANCIACION_NO_ENLAZADA,0,9,47,38,1,0,95
SIN_PAGOS,0,0,0,3,0,0,3


## 2. Embudo de población y conteos por objetivo

La población primaria son los créditos de **libranza**. Los de libre inversión tienen otro mecanismo de pago (no hay descuento por nómina) y mezclarlos inflaba el AUC de la v1. Un crédito es elegible si su ventana de 12 meses está madura, no es indeterminado y no pertenece a la cohorte piloto de 2021.

In [3]:
lib = t[t["producto"] == "libranza"]
pasos = [
    ("Créditos en bd_credito", len(R["tablas"]["credito"]), None),
    ("(−) anulados o pendientes de desembolso", -(len(R["tablas"]["credito"]) - len(t)), None),
    ("= créditos no anulados", len(t), None),
    ("(−) libre inversión", -int((t["producto"] != "libranza").sum()), None),
    ("= libranza no anulados", len(lib), None),
    ("(−) ventana de 12 meses inmadura", -int((lib["motivo_exclusion"] == "VENTANA_INMADURA").sum()), None),
    ("(−) indeterminados (con saldo, sin crédito posterior)", -int((lib["motivo_exclusion"] == "INDETERMINADO").sum()), None),
    ("(−) cohorte 2021", -int((lib["motivo_exclusion"] == "COHORTE_2021").sum()), None),
    ("= población primaria (libranza elegible)", int(lib["elegible"].sum()), int(lib.loc[lib["elegible"], "y_90_12m"].sum())),
    ("   de ellos cohortes < 2025", int((lib["elegible"] & (lib["conjunto_temporal"] == "<2025")).sum()),
     int(lib.loc[lib["elegible"] & (lib["conjunto_temporal"] == "<2025"), "y_90_12m"].sum())),
    ("   de ellos cohortes 2025+", int((lib["elegible"] & (lib["conjunto_temporal"] == "2025+")).sum()),
     int(lib.loc[lib["elegible"] & (lib["conjunto_temporal"] == "2025+"), "y_90_12m"].sum())),
]
embudo = pd.DataFrame(pasos, columns=["paso", "creditos", "eventos_90_12m"])
io_datos.guardar_resultado(embudo, "fase1", "embudo_poblacion")
embudo

,paso,creditos,eventos_90_12m
0,Créditos en bd_credito,1267,NaN
1,(−) anulados o pendientes de desembolso,-106,NaN
2,= créditos no anulados,1161,NaN
3,(−) libre inversión,-74,NaN
4,= libranza no anulados,1087,NaN
5,(−) ventana de 12 meses inmadura,-310,NaN
6,"(−) indeterminados (con saldo, sin crédito pos...",-42,NaN
7,(−) cohorte 2021,-4,NaN
8,= población primaria (libranza elegible),731,21.0
9,de ellos cohortes < 2025,494,16.0


In [4]:
el = lib[lib["elegible"]]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
etiquetas = ["No anulados", "Libranza", "12 meses\nmaduros", "Sin\nindeterminados", "Elegibles\n(sin 2021)"]
n_madura = int(lib["madura_12m"].sum())
n_sin_ind = int((lib["madura_12m"] & (lib["cierre_v2"] != "INDETERMINADO")).sum())
valores = [len(t), len(lib), n_madura, n_sin_ind, len(el)]
barras = ax.bar(etiquetas, valores, color=[figuras.GRIS] * 4 + [figuras.AZUL], width=0.62)
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width() / 2, v + 15, f"{v:,}".replace(",", "."), ha="center", fontsize=9)
ax.text(4, len(el) / 2, f"{int(el['y_90_12m'].sum())} eventos\n90+ en 12 m\n({100 * el['y_90_12m'].mean():.1f} %)",
        ha="center", va="center", color="white", fontsize=8.5, fontweight="bold")
ax.set_ylabel("Créditos")
ax.set_title("Embudo de la población primaria")
ax.grid(axis="x", visible=False)
figuras.guardar(fig, "fase1", "F1_01", "embudo_poblacion",
                "De 1.161 créditos no anulados quedan 731 de libranza elegibles con 21 eventos 90+ en 12 meses.")

Ahora cuento los eventos para las tres definiciones del protocolo. Para 30+/6m la madurez exigida es de 6 meses, así que su población es más grande. Incluyo también la variante B de 90+/12m, que extiende la observación de los cierres sospechosos hasta la fecha de corte. Esa variante es una cota superior, no una alternativa realista.

In [5]:
def poblacion(df, evento):
    madura = "madura_6m" if evento == "y_30_6m" else "madura_12m"
    return df[df[madura] & (df["cierre_v2"] != "INDETERMINADO") & ~df["anio_cohorte"].isin(config.COHORTES_EXCLUIR)]


filas = []
for ev, desc in [("y_90_12m", "90+ en 12 meses (primario)"), ("y_60_12m", "60+ en 12 meses"),
                 ("y_30_6m", "30+ en 6 meses"), ("y_90_12m_B", "90+ en 12 meses, variante B (cota)")]:
    for nombre, sub in [("todos los productos", t), ("libranza", lib),
                        ("libranza < 2025", lib[lib["conjunto_temporal"] == "<2025"]),
                        ("libranza 2025+", lib[lib["conjunto_temporal"] == "2025+"])]:
        p = poblacion(sub, ev)
        filas.append({"objetivo": desc, "poblacion": nombre, "creditos": len(p), "eventos": int(p[ev].sum()),
                      "clientes": p["grupo_cliente"].nunique()})
conteos = pd.DataFrame(filas)
conteos["tasa_%"] = (100 * conteos["eventos"] / conteos["creditos"]).round(2)
lo, hi = est.wilson(conteos["eventos"], conteos["creditos"])
conteos["ic95_%"] = [f"{100 * a:.1f}–{100 * b:.1f}" for a, b in zip(lo, hi)]
io_datos.guardar_resultado(conteos, "fase1", "conteos_por_objetivo")
conteos.pivot_table(index="objetivo", columns="poblacion", values="eventos", sort=False)

poblacion,todos los productos,libranza,libranza < 2025,libranza 2025+
objetivo,,,,
90+ en 12 meses (primario),46.0,21.0,16.0,5.0
60+ en 12 meses,56.0,27.0,20.0,7.0
30+ en 6 meses,96.0,62.0,44.0,18.0
"90+ en 12 meses, variante B (cota)",152.0,124.0,118.0,6.0


In [6]:
conteos[conteos["poblacion"] == "libranza"]

,objetivo,poblacion,creditos,eventos,clientes,tasa_%,ic95_%
1,90+ en 12 meses (primario),libranza,731,21,356,2.87,1.9–4.4
5,60+ en 12 meses,libranza,731,27,356,3.69,2.6–5.3
9,30+ en 6 meses,libranza,788,62,371,7.87,6.2–10.0
13,"90+ en 12 meses, variante B (cota)",libranza,731,124,356,16.96,14.4–19.9


**Lectura.** Con el evento primario, la población de libranza tiene **21 eventos en 731 créditos (2,9 %; IC 95 % 1,9–4,4)**, repartidos en 356 grupos de cliente. El promedio es de dos créditos por cliente, lo que confirma que la validación tiene que agruparse por cliente. Los objetivos secundarios multiplican la información: 60+/12m suma algunos eventos más, y 30+/6m más que duplica el número, a costa de medir una mora más temprana y curable. Esta diferencia es la razón por la que el protocolo deja los objetivos auxiliares como sensibilidad y no como sustituto.

## 3. ¿">" o "≥" 90 días?

El código siempre usó `max_dpd_12m >= 90`, mientras el texto de la tesis decía "más de 90 días". Si ningún crédito tiene exactamente 90 días como máximo, las dos definiciones coinciden y basta con corregir el texto.

In [7]:
p90 = poblacion(t, "y_90_12m")
exacto = int((p90["max_dpd_12m"] == 90).sum())
print(f"Créditos elegibles (todos los productos) con máximo DPD en 12 meses exactamente igual a 90: {exacto}")
print("Máximo DPD de los eventos 90+ de libranza (percentiles):",
      el.loc[el["y_90_12m"] == 1, "max_dpd_12m"].quantile([0, .25, .5, .75, 1]).to_dict())

Créditos elegibles (todos los productos) con máximo DPD en 12 meses exactamente igual a 90: 0
Máximo DPD de los eventos 90+ de libranza (percentiles): {0.0: 95.0, 0.25: 105.0, 0.5: 130.0, 0.75: 199.0, 1.0: 313.0}


Ningún crédito queda exactamente en 90, así que la definición operativa no cambia. **Decisión:** el texto de la tesis dirá "90 días o más", en línea con el código y con la forma en que la SFC establece las alturas de mora para consumo.

## 4. Riesgos que compiten con el evento

Un crédito prepagado o refinanciado a los tres meses cuenta como "no evento" en la ventana de 12 meses, aunque solo lo observé tres meses. Es una forma de riesgo competitivo. No sesga la etiqueta en sí, porque el crédito ya no puede caer en mora, pero sí cambia lo que la tasa significa. Por eso cuantifico cuántos elegibles tienen menos de 12 meses de observación real.

In [8]:
el_obs = el.assign(meses_obs=el["dias_obs_12m"] / 30.44)
el_obs["tramo_obs"] = pd.cut(el_obs["meses_obs"], [-1, 3, 6, 9, 11.9, 13], labels=["≤3 m", "3–6 m", "6–9 m", "9–12 m", "12 m"])
obs = pd.crosstab(el_obs["cierre_v2"], el_obs["tramo_obs"], margins=True, margins_name="Total")
io_datos.guardar_resultado(obs.reset_index(), "fase1", "meses_observados_elegibles")
print(f"Elegibles de libranza con menos de 12 meses observados: {(el_obs['meses_obs'] < 11.9).sum()} "
      f"({100 * (el_obs['meses_obs'] < 11.9).mean():.1f} %)")
obs

Elegibles de libranza con menos de 12 meses observados: 519 (71.0 %)


tramo_obs,≤3 m,3–6 m,6–9 m,9–12 m,12 m,Total
cierre_v2,,,,,,
ACTIVO,0,0,0,0,56,56
A_TERMINO,20,21,40,25,60,166
CIERRE_ANTICIPADO_SIN_EXPLICACION,3,2,0,0,0,5
CON_SALDO_CLIENTE_RECURRENTE,10,27,18,15,12,82
PREPAGO,37,46,36,31,35,185
REFINANCIACION,3,37,48,25,31,144
REFINANCIACION_NO_ENLAZADA,17,28,13,17,15,90
SIN_PAGOS,0,0,0,0,3,3
Total,90,161,155,113,212,731


**Lectura.** El 71 % de los elegibles de libranza salió de la ventana antes de los 12 meses, sobre todo por prepago, refinanciación (enlazada o no) y pago a término de créditos cortos. Solo 212 créditos se observaron los 12 meses completos. En libranza esto es normal: el cliente amplía el monto o cancela con la prima. La consecuencia metodológica es doble. Primero, la tasa de 2,9 % es una tasa *entre quienes siguieron expuestos o cerraron*, no una probabilidad marginal a 12 meses. Segundo, el modelo de supervivencia (sección 9), que trata esos cierres como censura, es el complemento natural. Además, una refinanciación hecha con mora puede ocultar un incumplimiento (*evergreening*); lo reviso en la sección 7.

## 5. Indeterminados y créditos marcados para verificación

Un indeterminado es un crédito que figura como finalizado con saldo pendiente y cuyo cliente no volvió a tomar un crédito. No sé si pagó por fuera del sistema, si se castigó o si es un error de registro. Por eso se excluye del análisis primario y reporto cotas.

In [9]:
ind = lib[(lib["cierre_v2"] == "INDETERMINADO") & lib["madura_12m"] & ~lib["anio_cohorte"].isin(config.COHORTES_EXCLUIR)]
print(f"Indeterminados de libranza con ventana madura: {len(ind)}")
print("Por cohorte:", ind["anio_cohorte"].value_counts().sort_index().to_dict())
tramos = [-1, 0, 29, 59, 89, 10 ** 6]
etq = ["0", "1–29", "30–59", "60–89", "90+"]
dist_ind = pd.DataFrame({
    "variante_A (hasta el último pago)": pd.cut(ind["max_dpd_12m"], tramos, labels=etq).value_counts().sort_index(),
    "variante_B (hasta el corte)": pd.cut(ind["max_dpd_12m_B"], tramos, labels=etq).value_counts().sort_index()})
io_datos.guardar_resultado(dist_ind.reset_index(names="max_dpd_12m"), "fase1", "indeterminados_distribucion_dpd")
dist_ind

Indeterminados de libranza con ventana madura: 42
Por cohorte: {np.int64(2022): 5, np.int64(2023): 11, np.int64(2024): 25, np.int64(2025): 1}


,variante_A (hasta el último pago),variante_B (hasta el corte)
0,18,6
1–29,14,7
30–59,4,2
60–89,4,2
90+,2,25


In [10]:
cotas = objetivo.cotas_indeterminados(t, "y_90_12m", filtro=lambda d: d["producto"] == "libranza")
io_datos.guardar_resultado(cotas, "fase1", "cotas_indeterminados_90_12m")
cotas

,escenario,creditos,eventos,tasa_%
0,Indeterminados excluidos (principal),731,21,2.87
1,Indeterminados = no evento (cota inferior),773,21,2.72
2,Indeterminados según su mora observada,773,23,2.98
3,Indeterminados = evento (cota superior),773,63,8.15


**Lectura.** Bajo la variante A, que deja de mirar en el último pago, casi todos los indeterminados están al día. Bajo la variante B, que sigue contando días hasta el corte, casi todos superan los 90 días: dejaron de pagar con saldo y nadie volvió a registrar nada. La verdad está en algún punto intermedio, y por eso las cotas de la tasa van de 2,7 % a más de 8 %. Es el supuesto más influyente de toda la etiqueta. En la Fase 4 voy a recalcular las métricas del modelo bajo los escenarios extremos.

Dos indeterminados ya superan 90 días **dentro** de la variante A: son los créditos 131 y 250. Los marco para que yo los verifique con la operación. Mientras tanto, siguen excluidos como el resto de indeterminados, tal como se decidió.

Hay un argumento lógico que conviene dejar escrito. La indeterminación se refiere a lo que pasó *después* del último pago, pero estos dos créditos alcanzaron 113 y 218 días de mora **antes** de ese punto y dentro de los 12 meses. Si la reconstrucción de sus pagos es correcta, su etiqueta ya está determinada: son eventos, pase lo que pase después. El 250 pagó incluso más que el plan, lo que sugiere que se puso al día tarde, pero eso no borra los 218 días de mora. Si la operación confirma que los pagos registrados son completos, propongo reclasificarlos como eventos en el análisis primario (23 eventos en 733 créditos). Lo dejo como decisión pendiente de Eider.

In [11]:
verif = t[t["cid"].isin(config.CREDITOS_VERIFICAR)][[
    "cid", "producto", "anio_cohorte", "cierre", "cierre_v2", "max_dpd_12m", "max_dpd_12m_B", "dias_hasta_90",
    "ratio_pagado_plan", "n_pagos", "madura_12m", "motivo_exclusion"]].copy()
verif["ratio_pagado_plan"] = verif["ratio_pagado_plan"].round(2)
verif["accion"] = "Verificar con la operación: ¿castigo, pago por fuera o error de registro?"
io_datos.guardar_resultado(verif, "fase1", "creditos_131_250_para_verificar")
verif

[Salida retirada del repositorio público: mostraba filas de créditos individuales.]


## 6. Comparación con el objetivo de la v1

Si mi reimplementación es correcta, debe reproducir el objetivo de la v1 crédito a crédito con el mismo export y los mismos parámetros. Cualquier diferencia sería un error de una de las dos versiones.

In [12]:
v1 = io_datos.leer_parquet(config.OBJETIVO_V1)[["cid", "cierre_v1", "max_dpd_12m", "y_90_12m", "y_60_12m", "elegible"]]
v1 = v1.rename(columns=lambda c: c if c == "cid" else f"{c}__v1")
m = t.merge(v1, on="cid", how="outer", indicator=True)
comp = pd.DataFrame([
    {"chequeo": "Créditos presentes en ambas versiones", "coincidencias": int((m["_merge"] == "both").sum()), "total": len(m)},
    {"chequeo": "Tipo de cierre (D3-D5)", "coincidencias": int((m["cierre_v2"] == m["cierre_v1__v1"]).sum()), "total": len(m)},
    {"chequeo": "Máximo DPD en 12 meses", "coincidencias": int((m["max_dpd_12m"] == m["max_dpd_12m__v1"]).sum()), "total": len(m)},
    {"chequeo": "Elegibilidad", "coincidencias": int((m["elegible"] == m["elegible__v1"]).sum()), "total": len(m)},
    {"chequeo": "y_90_12m (incluye faltantes)", "coincidencias": int((m["y_90_12m"].fillna(-1) == m["y_90_12m__v1"].fillna(-1)).sum()), "total": len(m)},
    {"chequeo": "y_60_12m (incluye faltantes)", "coincidencias": int((m["y_60_12m"].fillna(-1) == m["y_60_12m__v1"].fillna(-1)).sum()), "total": len(m)},
])
comp["%"] = (100 * comp["coincidencias"] / comp["total"]).round(2)
io_datos.guardar_resultado(comp, "fase1", "comparacion_objetivo_v1")
comp

,chequeo,coincidencias,total,%
0,Créditos presentes en ambas versiones,1161,1161,100.0
1,Tipo de cierre (D3-D5),1161,1161,100.0
2,Máximo DPD en 12 meses,1161,1161,100.0
3,Elegibilidad,1161,1161,100.0
4,y_90_12m (incluye faltantes),1161,1161,100.0
5,y_60_12m (incluye faltantes),1161,1161,100.0


La reconstrucción reproduce la v1 al 100 % en todos los chequeos. Eso **no** prueba que el método sea correcto, porque ambas versiones comparten los mismos supuestos. Sí descarta errores de implementación. La validez del método la contrasto con una fuente independiente: `bd_pagosempresas`, donde el empleador reporta el valor en mora de cada recaudo.

In [13]:
pe = R["tablas"]["pagosempresas"] if "pagosempresas" in R["tablas"] else io_datos.leer_tabla("pagosempresas")
pe = pe[pe["estado"] == "ACTIVO"].assign(cid=llave(pe["credito_id"]))
vc = numero(pe["valorcuota"])
pe["cuotas_mora"] = numero(pe["valormora"]) / vc.where(vc > 0)
vpe = t.set_index("cid").join(pe.groupby("cid")["cuotas_mora"].max(), how="inner")
a, b = (vpe["cuotas_mora"] >= 1), (vpe["max_dpd_A"] >= 30)
kappa = cohen_kappa_score(a, b)
tabla_pe = pd.crosstab(a.rename("empleador reporta ≥ 1 cuota en mora"), b.rename("reconstrucción: DPD ≥ 30"), margins=True)
print(f"Créditos con reporte del empleador: {len(vpe)} | acuerdo: {100 * (a == b).mean():.1f} % | kappa de Cohen: {kappa:.3f}")
io_datos.guardar_resultado(tabla_pe.reset_index(), "fase1", "validacion_pagosempresas")
tabla_pe

Créditos con reporte del empleador: 225 | acuerdo: 88.0 % | kappa de Cohen: 0.466


reconstrucción: DPD ≥ 30,False,True,All
empleador reporta ≥ 1 cuota en mora,,,
False,183,23,206
True,4,15,19
All,187,38,225


**Lectura.** El acuerdo bruto es alto, pero el kappa, que corrige por el acuerdo esperado al azar, es moderado. El desacuerdo dominante son créditos con 30+ días reconstruidos que el empleador no reporta en mora. Tiene sentido: `bd_pagosempresas` solo cubre el recaudo empresarial hasta septiembre de 2025 y no ve la mora de los clientes que pagan directamente. (En la v1 este kappa salió como `nan` por un error de cálculo; aquí queda corregido.) La validación definitiva sigue siendo la revisión manual de la muestra de la sección 10.

## 7. Refinanciación con mora (*evergreening*)

Si un crédito en mora se refinancia, la refinanciación "limpia" la mora: el crédito viejo se cierra sin evento y el nuevo empieza al día. Reviso a qué altura de mora se refinanciaron los créditos con enlace explícito.

In [14]:
rf = lib[lib["cierre_v2"] == "REFINANCIACION"]
dist_rf = pd.cut(rf["dpd_al_refinanciar"], tramos, labels=etq).value_counts().sort_index()
io_datos.guardar_resultado(dist_rf.reset_index(name="creditos").rename(columns={"index": "dpd_al_refinanciar"}),
                           "fase1", "dpd_al_refinanciar")
print(f"Refinanciaciones de libranza: {len(rf)}")
dist_rf

Refinanciaciones de libranza: 144


dpd_al_refinanciar
0        128
1–29      13
30–59      2
60–89      0
90+        1
Name: count, dtype: int64

La gran mayoría se refinancia al día o con mora menor de 30 días, lo que respalda la regla D3 (refinanciación = ampliación de monto, no rescate). Los pocos casos con 30+ días no alcanzan a cambiar el conteo de eventos de forma material, y quedan documentados como limitación.

## 8. Sensibilidad de la etiqueta

Dos parámetros de la reconstrucción son discrecionales: la **tolerancia** con la que una cuota se considera pagada y la **ventana** de la regla D4 (refinanciación no enlazada). Mido cuánto cambian la población primaria y los eventos al moverlos.

In [15]:
sens = []
for etiqueta, kw in [("Principal (tol 5 %, D4 −15/+45)", {}),
                     ("Tolerancia 0 %", {"tol_rel": 0.0}), ("Tolerancia 10 %", {"tol_rel": 0.10}),
                     ("D4 ventana −7/+30", {"ventana_refi": (-7, 30)}), ("D4 ventana −30/+60", {"ventana_refi": (-30, 60)}),
                     ("Sin regla D4", {"ventana_refi": (0, -1)})]:
    ts = objetivo.cargar_y_reconstruir(**kw)["objetivo"]
    ls = ts[ts["producto"] == "libranza"]
    e = ls[ls["elegible"]]
    sens.append({"escenario": etiqueta, "elegibles": len(e), "eventos_90_12m": int(e["y_90_12m"].sum()),
                 "eventos_60_12m": int(e["y_60_12m"].sum()),
                 "indeterminados_maduros": int(((ls["cierre_v2"] == "INDETERMINADO") & ls["madura_12m"]).sum())})
sens = pd.DataFrame(sens)
sens["tasa_90_%"] = (100 * sens["eventos_90_12m"] / sens["elegibles"]).round(2)
io_datos.guardar_resultado(sens, "fase1", "sensibilidad_etiqueta")
sens

,escenario,elegibles,eventos_90_12m,eventos_60_12m,indeterminados_maduros,tasa_90_%
0,"Principal (tol 5 %, D4 −15/+45)",731,21,27,42,2.87
1,Tolerancia 0 %,731,21,27,42,2.87
2,Tolerancia 10 %,731,21,27,42,2.87
3,D4 ventana −7/+30,721,21,27,52,2.91
4,D4 ventana −30/+60,743,22,28,30,2.96
5,Sin regla D4,711,20,26,62,2.81


**Lectura.** El evento primario es robusto a la tolerancia: los créditos que llegan a 90 días deben varias cuotas completas, no centavos. La regla D4 sí mueve créditos entre "indeterminado" y "refinanciación no enlazada", es decir, entre excluido e incluido sin evento, pero casi no cambia el número de eventos. Sin la regla, los indeterminados crecen de forma visible. Esto confirma que la incertidumbre de la etiqueta está concentrada en los indeterminados y no en la reconstrucción de la mora.

## 9. Objetivo de supervivencia

Para no desperdiciar los créditos con ventana inmadura, construyo también el tiempo hasta el primer día con DPD ≥ 90, censurado en el fin de observación. Incluye todos los créditos de libranza no anulados de 2022 en adelante. Los indeterminados entran censurados en su último pago, que es el supuesto de la variante A, y su sensibilidad se trata como en la etiqueta binaria.

In [16]:
sv = lib[~lib["anio_cohorte"].isin(config.COHORTES_EXCLUIR)]
res_sv = pd.DataFrame([{
    "creditos": len(sv), "eventos_90": int(sv["evento_90"].sum()), "censurados": int((sv["evento_90"] == 0).sum()),
    "de_ellos_inmaduros_12m": int((~sv["madura_12m"]).sum()),
    "eventos_90_en_inmaduros": int(sv.loc[~sv["madura_12m"], "evento_90"].sum()),
    "seguimiento_mediano_meses": round(float(sv["tiempo_90"].median() / 30.44), 1),
    "clientes": sv["grupo_cliente"].nunique()}])
io_datos.guardar_resultado(res_sv, "fase1", "resumen_supervivencia")
res_sv

,creditos,eventos_90,censurados,de_ellos_inmaduros_12m,eventos_90_en_inmaduros,seguimiento_mediano_meses,clientes
0,1083,29,1054,310,1,6.1,574


In [17]:
km = est.kaplan_meier(sv["tiempo_90"], sv["evento_90"], grilla=np.arange(0, 731, 30.44 / 2))
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.step(km["t"] / 30.44, 100 * km["incidencia"], where="post", color=figuras.AZUL, lw=1.8)
ax.fill_between(km["t"] / 30.44, 100 * km["inc_inf"], 100 * km["inc_sup"], step="post", color=figuras.AZUL, alpha=0.15)
ax.axvline(12, color=figuras.GRIS, ls="--", lw=0.8)
i12 = km.iloc[(km["t"] - 365.25).abs().argmin()]
ax.annotate(f"12 meses: {100 * i12['incidencia']:.1f} %\n(IC 95 % {100 * i12['inc_inf']:.1f}–{100 * i12['inc_sup']:.1f})",
            xy=(12, 100 * i12["incidencia"]), xytext=(13.5, 100 * i12["incidencia"] * 0.45), fontsize=8,
            arrowprops=dict(arrowstyle="-", color=figuras.GRIS))
ax.set_xlabel("Meses desde el desembolso")
ax.set_ylabel("Incidencia acumulada de 90+ (%)")
ax.set_title("Tiempo hasta 90+ días de mora — libranza (Kaplan-Meier)")
ax.set_xlim(0, 24)
figuras.guardar(fig, "fase1", "F1_02", "km_tiempo_90",
                "La incidencia acumulada de 90+ a 12 meses con censura es mayor que la tasa binaria porque no cuenta los cierres tempranos como éxitos.")

**Lectura.** La incidencia de Kaplan-Meier a 12 meses es de **5,7 % (IC 95 % 3,7–8,7)**, el doble de la tasa binaria del 2,9 %. La diferencia es justamente el efecto de los riesgos competitivos de la sección 4: el estimador de supervivencia no trata un crédito prepagado al tercer mes como si hubiera sobrevivido 12 meses. Hay que leerla con cuidado, porque bajo riesgos competitivos 1 − KM sobreestima la incidencia acumulada real. Las dos cifras acotan el rango plausible del riesgo a 12 meses, y la discusión de qué probabilidad debe estimar el modelo (marginal o condicionada a seguir expuesto) entra en el capítulo de métodos.

Un matiz importante para el diseño: el objetivo de supervivencia suma exposición (1.083 créditos frente a 731), pero **casi no suma eventos**. De los 310 créditos inmaduros, solo uno ya llegó a 90+. Su valor está en usar el tiempo de exposición de todos los créditos, no en añadir incumplimientos.

## 10. Muestra estratificada para validación manual

La validación definitiva de la etiqueta es que yo revise con la operación una muestra de créditos contra sus extractos. Selecciono 30 créditos al azar (semilla fija) por estratos que cubren los casos difíciles. La lista de identificadores queda en `_privado/` porque es para uso interno; aquí solo muestro cuántos hay por estrato.

In [18]:
rng = np.random.default_rng(config.SEMILLA)
base_m = lib[lib["madura_12m"]].copy()
base_m["estrato"] = np.select(
    [base_m["cid"].isin(config.CREDITOS_VERIFICAR), base_m["cierre_v2"] == "INDETERMINADO", base_m["max_dpd_12m"] >= 90,
     base_m["max_dpd_12m"] >= 30, base_m["cierre_v2"].isin(["REFINANCIACION", "REFINANCIACION_NO_ENLAZADA"])],
    ["marcado_131_250", "indeterminado", "evento_90", "mora_30_89", "refinanciado_sin_evento"], "al_dia_sin_evento")
cuota = {"marcado_131_250": 2, "indeterminado": 5, "evento_90": 8, "mora_30_89": 5, "refinanciado_sin_evento": 5,
         "al_dia_sin_evento": 5}
muestra = pd.concat([g.sample(min(cuota[k], len(g)), random_state=config.SEMILLA) for k, g in base_m.groupby("estrato")])
io_datos.guardar_privado(muestra[["cid", "estrato", "anio_cohorte", "cierre_v2", "max_dpd_12m", "y_90_12m"]],
                         "muestra_validacion_manual")
muestra["estrato"].value_counts().rename("creditos_en_muestra").to_frame()

,creditos_en_muestra
estrato,
evento_90,8
al_dia_sin_evento,5
indeterminado,5
mora_30_89,5
refinanciado_sin_evento,5
marcado_131_250,2


## 11. Guardado del objetivo v2

In [19]:
cols = ["cid", "cli", "grupo_cliente", "producto", "anio_cohorte", "conjunto_temporal", "t0", "t_ref", "estado",
        "cnfrecaudo", "cierre", "cierre_v2", "fin_obs_A", "f_cierre", "dias_obs_A", "dias_obs_12m", "max_dpd_A",
        "max_dpd_B", "max_dpd_6m", "max_dpd_12m", "max_dpd_12m_B", "dias_hasta_30", "dias_hasta_60", "dias_hasta_90",
        "dias_hasta_90_B", "madura_6m", "madura_12m", "y_90_12m", "y_60_12m", "y_30_6m", "y_90_12m_B",
        "motivo_exclusion", "elegible", "evento_90", "tiempo_90", "refi_no_enlazada", "credito_posterior",
        "dpd_al_refinanciar", "ratio_pagado_plan"]
ruta = io_datos.guardar_privado(t[cols], "objetivo_v2")
reg = io_datos.escribir_registro("02_objetivo_y_poblacion", {
    "fecha_corte": str(R["corte"].date()), "parametros_mora": config.PARAMS_MORA, "ventana_refi": config.VENTANA_REFI,
    "libranza_elegibles": len(el), "eventos_90_12m": int(el["y_90_12m"].sum()),
    "eventos_60_12m": int(el["y_60_12m"].sum()),
    "coincidencia_v1_y90": float(comp.loc[comp["chequeo"].str.startswith("y_90"), "%"].iloc[0]),
    "kappa_pagosempresas": round(float(kappa), 3),
    "salida_privada": {"archivo": "_privado/objetivo_v2.parquet", "sha256": io_datos.sha256(ruta)}})
figuras.escribir_catalogo("fase1", "Figuras de la Fase 1")
print("Guardado:", ruta.name, "| registro:", reg.name)

Guardado: objetivo_v2.parquet | registro: registro_02_objetivo_y_poblacion.json


## Conclusiones

- **El objetivo es reproducible:** la reimplementación coincide al 100 % con la v1. La definición operativa es **≥ 90 días** y ningún crédito queda exactamente en 90, así que solo hay que corregir el texto.
- **Población primaria:** 731 créditos de libranza con 21 eventos 90+/12m (2,9 %): 494/16 en cohortes < 2025 y 237/5 en 2025+.
- **La mayor fuente de incertidumbre de la etiqueta son los indeterminados.** Según cómo se traten, la tasa va de 2,7 % a más de 8 %. Los créditos 131 y 250 quedan marcados para verificación con la operación.
- **Créditos 131 y 250:** alcanzaron 113 y 218 días de mora dentro de su ventana observada. Lógicamente son eventos determinados. Propongo reclasificarlos si la operación confirma sus pagos (decisión de Eider).
- **Riesgos competitivos:** el 71 % de los elegibles salió de la ventana antes de 12 meses por prepago, refinanciación o pago a término. La incidencia de Kaplan-Meier a 12 meses (5,7 %) duplica la tasa binaria (2,9 %). El modelo de supervivencia suma exposición pero casi ningún evento nuevo (1 entre 310 inmaduros).
- La validación externa con `bd_pagosempresas` da un acuerdo del 88 % y un kappa de 0,47 (moderado), con la limitación de cobertura descrita.
- La etiqueta es robusta a la tolerancia de pago y casi no depende de la ventana D4 en número de eventos.